# Operaciones puntuales y transformaciones sobre imágenes

Este notebook implementa las principales **operaciones puntuales** (identidad,
negativo, umbral, intervalo umbral binario, umbral en escala de grises,
extensión de niveles de gris y reducción de niveles de gris) y **transformaciones**
(adición, sustracción y rodaja de planos de bits).

Las funciones trabajan sobre arrays `numpy`, por lo que son independientes del
formato de archivo (TIFF, PNG, JPG, ...). Trabajamos con dos imágenes **TIFF en
escala de grises de 8 bits**:

- `img/Fig0418(a)(ray_traced_bottle_original).tif` — botella renderizada (ray tracing)
- `img/Fig0462(a)(PET_image).tif` — imagen PET

Como tienen tamaños distintos, la segunda se **redimensiona** al tamaño de la
primera para poder sumarla/restarla.

## 1. Preparación

Importamos las librerías, cargamos las imágenes y definimos un helper de
visualización que normaliza el rango mostrado al mínimo/máximo real de cada
imagen (así se ven correctamente archivos de cualquier profundidad de bits).

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

RUTA_1 = "img/Fig0418(a)(ray_traced_bottle_original).tif"
RUTA_2 = "img/Fig0462(a)(PET_image).tif"


def cargar(ruta, tamano=None):
    """Carga una imagen en grises de 8 bits (uint8).

    Si `tamano` se indica, la redimensiona a ese tamaño. Es necesario
    para que las transformaciones (adición/sustracción) trabajen con
    imágenes del mismo tamaño.
    """
    imagen = Image.open(ruta)
    if imagen.mode != "L":
        imagen = imagen.convert("L")
    if tamano is not None and imagen.size != tamano:
        imagen = imagen.resize(tamano, Image.LANCZOS)
    return np.asarray(imagen).astype(np.uint8)


imagen_1 = cargar(RUTA_1)
imagen_2 = cargar(RUTA_2, tamano=imagen_1.shape[::-1])

print(f"Imagen 1: {imagen_1.shape} | dtype: {imagen_1.dtype} | rango: {imagen_1.min()}-{imagen_1.max()}")
print(f"Imagen 2: {imagen_2.shape} | dtype: {imagen_2.dtype} | rango: {imagen_2.min()}-{imagen_2.max()} (redimensionada a {imagen_1.shape[1]}x{imagen_1.shape[0]})")

In [ ]:
def niveles(dtype):
    """Número total de niveles de gris `L` según el dtype (256 para uint8)."""
    return int(2 ** np.iinfo(dtype).bits)


def mostrar(imagenes, titulos=None, figsize=None, cmap="gray", normalizar=True):
    """Muestra una lista de imágenes en una fila.

    Por defecto normaliza `vmin`/`vmax` al mínimo/máximo real de cada imagen
    para que cualquier profundidad se visualice correctamente.
    """
    if not isinstance(imagenes, (list, tuple)):
        imagenes = [imagenes]
    n = len(imagenes)
    figsize = figsize or (4 * n, 4)
    _, ejes = plt.subplots(1, n, figsize=figsize)
    if n == 1:
        ejes = [ejes]
    for i, img in enumerate(imagenes):
        arr = np.asarray(img)
        vmin, vmax = None, None
        if normalizar:
            vmin = float(np.nanmin(arr))
            vmax = float(np.nanmax(arr))
            if vmin == vmax:
                vmax = vmin + 1.0
        ejes[i].imshow(arr, cmap=cmap if arr.ndim == 2 else None, vmin=vmin, vmax=vmax)
        ejes[i].axis("off")
        if titulos and i < len(titulos):
            ejes[i].set_title(titulos[i])
    plt.tight_layout()
    plt.show()

## 2. Operaciones puntuales

Las operaciones puntuales transforman cada píxel de forma independiente:
$s = T(r)$, donde $r$ es el valor de entrada y $s$ el de salida. Se definen en
escala de grises; para imágenes en color se aplican a cada canal por separado
(véase la sección 2.8).

### 2.1 Identidad

$s = r$. Devuelve la imagen tal cual; sirve como referencia para comparar.

In [ ]:
def identidad(arr):
    """Identidad: s = r. Devuelve la imagen sin cambios."""
    return np.array(arr)


mostrar([imagen_1, identidad(imagen_1)], titulos=["Original", "Identidad"])

### 2.2 Inverso o negativo

$s = L - 1 - r$, con $L$ el número de niveles de gris (256 para imágenes de 8
bits). Invierte los tonos de la imagen.

In [ ]:
def negativo(arr):
    """Inverso: s = L - 1 - r."""
    maximo = niveles(arr.dtype) - 1
    return (maximo - arr.astype(np.int32)).astype(arr.dtype)


mostrar([imagen_1, negativo(imagen_1)], titulos=["Original", "Negativo"])

### 2.3 Umbral

$s = 0$ si $r < T$ y $s = L-1$ (blanco) en caso contrario. Produce una imagen
binaria que separa la imagen en dos clases según la intensidad.

In [ ]:
def umbral(arr, t):
    """Umbral binario: 0 si r < t, L-1 en caso contrario."""
    maximo = niveles(arr.dtype) - 1
    return np.where(arr < t, 0, maximo).astype(arr.dtype)


T = 128
mostrar([imagen_1, umbral(imagen_1, T)], titulos=["Original", f"Umbral (T={T})"])

### 2.4 Intervalo umbral binario

$s = L-1$ si $a \leq r \leq b$ y $s = 0$ en el resto. Resalta en blanco solo la
zona cuyos niveles están dentro del intervalo $[a, b]$.

In [ ]:
def umbral_intervalo_binario(arr, a, b):
    """Intervalo umbral binario: L-1 si a <= r <= b, 0 en el resto."""
    maximo = niveles(arr.dtype) - 1
    return np.where((arr >= a) & (arr <= b), maximo, 0).astype(arr.dtype)


a, b = 100, 200
mostrar([imagen_1, umbral_intervalo_binario(imagen_1, a, b)],
        titulos=["Original", f"Intervalo binario [{a}, {b}]"])

### 2.5 Umbral en escala de grises

Variante del intervalo anterior: dentro de $[a, b]$ se **conserva el valor de
gris original** y fuera se pone $0$ (fondo negro). Útil para segmentar una zona
de interés manteniendo su detalle interno.

In [ ]:
def umbral_escala_grises(arr, a, b):
    """Conserva los grises dentro de [a, b] y pone 0 fuera del intervalo."""
    return np.where((arr >= a) & (arr <= b), arr, 0).astype(arr.dtype)


mostrar([imagen_1, umbral_escala_grises(imagen_1, a, b)],
        titulos=["Original", f"Escala de grises [{a}, {b}]"])

### 2.6 Extensión de niveles de gris

Estiramiento lineal de contraste: mapea el rango $[r_{min}, r_{max}]$ al rango
completo $[0, L-1]$. Con los valores por defecto (mínimo y máximo de la imagen)
aprovecha todo el rango dinámico; también se puede fijar un intervalo manual
(por ejemplo los percentiles) para no amplificar ruido.

In [ ]:
def extension_niveles_gris(arr, minimo=None, maximo=None):
    """Extensión lineal [minimo, maximo] -> [0, L-1]."""
    maximo_out = niveles(arr.dtype) - 1
    rmin = arr.min() if minimo is None else minimo
    rmax = arr.max() if maximo is None else maximo
    if rmax <= rmin:
        return np.array(arr)
    escala = maximo_out / (rmax - rmin)
    estirada = (arr.astype(np.float64) - rmin) * escala
    return np.clip(estirada, 0, maximo_out).astype(arr.dtype)


mostrar([imagen_2, extension_niveles_gris(imagen_2)],
        titulos=["PET original (oscura)", "Extensión al rango completo"])

### 2.7 Reducción de niveles de gris

Cuantiza los valores a $2^{bits}$ niveles y los **re-expande al rango completo**
$[0, L-1]$. La imagen se *posteriza* (aparecen bandas de tono) pero no se
oscurece. Con 1 bit se obtiene directamente un umbral binario

In [ ]:
def reduccion_niveles_gris(arr, bits):
    """Cuantiza a 2^bits niveles y re-expande al rango completo [0, L-1]."""
    maximo = niveles(arr.dtype) - 1
    if bits >= int(np.log2(niveles(arr.dtype))):
        return np.array(arr)
    cuantos = (1 << bits) - 1
    arr_f = arr.astype(np.float64) / maximo
    cuantizada = np.round(arr_f * cuantos) / cuantos
    return (cuantizada * maximo).astype(arr.dtype)


mostrar([imagen_1,
         reduccion_niveles_gris(imagen_1, 4),
         reduccion_niveles_gris(imagen_1, 1)],
        titulos=["Original", "Reducción a 4 bits (16 niveles)", "Reducción a 1 bit (2 niveles)"])

### 2.8 Aplicación por canal (imágenes en color)

Una imagen en color RGB se interpreta como tres matrices (rojo, verde, azul).
Como las operaciones puntuales actúan píxel a píxel, se aplican a **cada canal
por separado**.

Lo demostramos con una imagen RGB sintética generada con `numpy`: canal R con
gradiente horizontal, canal G con gradiente vertical y canal B constante.

In [ ]:
def aplicar_por_canal(arr, funcion, **kwargs):
    """Aplica una operación puntual a cada canal de una imagen en color."""
    canales = [funcion(arr[..., c], **kwargs) for c in range(arr.shape[-1])]
    return np.stack(canales, axis=-1)


alto, ancho = 256, 256
xx, _ = np.meshgrid(np.linspace(0, 255, ancho), np.linspace(0, 255, alto))
_, yy = np.meshgrid(np.linspace(0, 255, ancho), np.linspace(0, 255, alto))
sintetica = np.stack([
    xx.astype(np.uint8),
    yy.astype(np.uint8),
    np.full((alto, ancho), 128, np.uint8),
], axis=-1)

negativa = aplicar_por_canal(sintetica, negativo)
binaria = aplicar_por_canal(sintetica, umbral, t=T)
mostrar([sintetica, negativa, binaria],
        titulos=["RGB sintética", "Negativo por canal", "Umbral por canal"],
        figsize=(14, 5))

## 3. Transformaciones entre imágenes

A diferencia de las operaciones puntuales (que usan una sola imagen), las
transformaciones combinan **dos imágenes** píxel a píxel, lo que exige que
tengan el mismo tamaño. Aquí la segunda imagen ya fue redimensionada en la
celda de preparación.

### 3.1 Adición

$s = r_1 + r_2$, recortada al rango válido $[0, L-1]$ para no desbordar.
Útil para composición o para sumar ruido controlado. (En el procesado de
secuencias, **promediar** varias imágenes reduce el ruido.)

In [ ]:
def adicion(a, b):
    """Suma de dos imágenes del mismo tamaño, recortada al rango válido."""
    maximo = niveles(a.dtype) - 1
    suma = a.astype(np.int32) + b.astype(np.int32)
    return np.clip(suma, 0, maximo).astype(a.dtype)


mostrar([imagen_1, imagen_2, adicion(imagen_1, imagen_2)],
        titulos=["Imagen 1", "Imagen 2", "Adición"],
        figsize=(14, 5))

### 3.2 Sustracción

$s = |r_1 - r_2|$. La diferencia con valor absoluto elimina el signo (no hay
niveles negativos) y resalta las **diferencias** entre dos imágenes: se usa
para detectar cambios, movimiento o regiones de interés.

In [ ]:
def sustraccion(a, b):
    """Valor absoluto de la diferencia de dos imágenes del mismo tamaño."""
    return np.abs(a.astype(np.int32) - b.astype(np.int32)).astype(a.dtype)


mostrar([imagen_1, imagen_2, sustraccion(imagen_1, imagen_2)],
        titulos=["Imagen 1", "Imagen 2", "|Sustracción|"],
        figsize=(14, 5))

### 3.3 Rodaja de planos de bits

Una imagen de 8 bits se puede descomponer en **8 planos binarios**, uno por bit
($0$ = menos significativo, $7$ = más significativo). El plano $p$ vale 255 en
los píxeles cuyo $p$-ésimo bit está encendido y 0 en el resto.

Los planos **superiores** concentran la información visual más importante;
en los **inferiores** aparecen detalles finos y ruido.

In [ ]:
def plano_de_bits(arr, p):
    """Extrae como binario (0/255) el plano de bit p (0 = menos significativo)."""
    return (((arr >> p) & 1) * 255).astype(np.uint8)


fig, ejes = plt.subplots(2, 4, figsize=(16, 9))
for p in range(8):
    ejes[p // 4, p % 4].imshow(plano_de_bits(imagen_1, p), cmap="gray", vmin=0, vmax=255)
    ejes[p // 4, p % 4].set_title(f"Plano {p} (bit {p})")
    ejes[p // 4, p % 4].axis("off")
plt.tight_layout()
plt.show()

In [ ]:
def reconstruir(planos):
    """Reconstruye la imagen original sumando los planos ponderados: p->2^p."""
    return sum((plano.astype(np.uint16) // 255) << p
               for p, plano in enumerate(planos)).astype(np.uint8)


planos = [plano_de_bits(imagen_1, p) for p in range(8)]
reconstruida = reconstruir(planos)

print("Reconstrucción correcta:", np.array_equal(reconstruida, imagen_1))
mostrar([imagen_1, reconstruida], titulos=["Original", "Reconstruida desde los 8 planos"])

## 4. Resumen

- Las **operaciones puntuales** mapean cada píxel de forma independiente
  ($s = T(r)$) y sirven para binarizar (umbrales), invertir (negativo),
  mejorar contraste (extensión) o comprimir la información (reducción).
- Para **color**, se aplican a cada canal por separado.
- Las **transformaciones** combinan dos imágenes (adición, sustracción) y
  exigen el mismo tamaño; la **rodaja de planos de bits** descompone la imagen
  en su información binaria y permite reconstruirla.

In [ ]:
from IPython.display import Image as DisplayImage
print("Fin del notebook.")